# Q2: ADASYN-Style Hard-Region Benchmark

This notebook starts Question 2 from `manuscript/experiment_plan.md`:

> Does MIMIC improve adaptive hard-region performance compared with ADASYN?

Reusable experiment machinery lives in `src/mimic_experiments/q2_adasyn.py`. This notebook chooses parameters, calls that module, and displays saved result tables and plots.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "mimic").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import GenerationPolicy

from mimic_experiments.q2_adasyn import (
    Q2Config,
    load_q2_dataset,
    load_q2_result_tables,
    plot_q2_metric_comparison,
    print_progress_event,
    published_reference_table,
    q2_dataset_registry,
    q2_result_table_manifest,
    q2_result_table_paths,
    run_q2_benchmark,
)


## Experiment Controls

Use `RUN_PROFILE = "run_full"` to execute and save the full repeated half-split experiment. Use `RUN_PROFILE = "view"` to skip fitting and regenerate displays from saved CSV tables.

In [2]:
# Dataset and run controls
DATASET = "pima"  # use a registry row number, or one of: "pima", "ionosphere", "abalone"
RUN_PROFILE = "run_full"  # "run_full" or "view"
RANDOM_STATE = 0
N_RUNS = 100
ARTIFACT_DIR = str(PROJECT_ROOT / "manuscript" / "artifacts" / "q2_adasyn")
WORKER_THREADS = 1

# MIMIC controls
MIMIC_MODE = "factorised"
MIMIC_CAPACITY_RUN_FULL = 0.25
MIMIC_FEATURE_N_JOBS = 1  # keep 1 when outer jobs parallelize work; increase when running one job at a time

# MIMIC generation policy controls
POLICY_METHOD = "smote"
POLICY_NEIGHBOUR_MODE = "normal"
POLICY_N_NEIGHBORS = 5
POLICY_LAMBDA_RANGE = (0.0, 1.0)

registry = q2_dataset_registry()
DATASET_KEY = registry.loc[DATASET, "key"] if isinstance(DATASET, int) else DATASET

config = Q2Config(
    dataset_key=DATASET_KEY,
    run_profile=RUN_PROFILE,
    random_state=RANDOM_STATE,
    n_runs=N_RUNS,
    artifact_dir=ARTIFACT_DIR,
    worker_threads=WORKER_THREADS,
    mimic_mode=MIMIC_MODE,
    mimic_capacity_run_full=MIMIC_CAPACITY_RUN_FULL,
    mimic_feature_n_jobs=MIMIC_FEATURE_N_JOBS,
    policy=GenerationPolicy(
        method=POLICY_METHOD,
        neighbour_mode=POLICY_NEIGHBOUR_MODE,
        n_neighbors=POLICY_N_NEIGHBORS,
        lambda_range=POLICY_LAMBDA_RANGE,
    ),
)
config


Q2Config(dataset_key='pima', run_profile='run_full', random_state=0, n_runs=100, mimic_mode='factorised', mimic_capacity_run_full=0.25, artifact_dir='/run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/MIMIC/manuscript/artifacts/q2_adasyn', worker_threads=1, mimic_feature_n_jobs=1, policy=GenerationPolicy(method='smote', neighbour_mode='normal', n_neighbors=5, lambda_range=(0.0, 1.0), class_conditioned=False, cluster_conditioned=False))

## Dataset Registry

The registry records the Q2 datasets from the ADASYN paper and which loaders are currently available.

In [3]:
display(
    q2_dataset_registry(
        artifact_dir=ARTIFACT_DIR,
        run_profile=RUN_PROFILE,
        mimic_mode=MIMIC_MODE,
        mimic_capacity=config.mimic_capacity,
        policy=config.policy,
        random_state=RANDOM_STATE,
    )
)


,key,dataset,status,experiment
0,vehicle,Vehicle,optional: exact ADASYN binary source needed,not run
1,pima,Pima Indian Diabetes,ready: OpenML data_id=37,not run
2,vowel,Vowel recognition,optional: exact ADASYN binary source needed,not run
3,ionosphere,Ionosphere,ready: OpenML name=ionosphere,not run
4,abalone,Abalone,ready: OpenML data_id=183; classes 18 vs 9,not run


## Published Reference Table

In [4]:
display(published_reference_table())


,dataset,method,oa,precision,recall,f_measure,g_mean
0,Vehicle,Decision tree,0.9220,0.8454,0.8199,0.8308,0.8834
1,Vehicle,SMOTE,0.9239,0.8236,0.8638,0.8418,0.9018
2,Vehicle,ADASYN,0.9257,0.8067,0.9015,0.8505,0.9168
3,Pima Indian Diabetes,Decision tree,0.6831,0.5460,0.5500,0.5469,0.6430
4,Pima Indian Diabetes,SMOTE,0.6557,0.5049,0.6201,0.5556,0.6454
5,Pima Indian Diabetes,ADASYN,0.6837,0.5412,0.6097,0.5726,0.6625
6,Vowel recognition,Decision tree,0.9760,0.8710,0.8700,0.8681,0.9256
7,Vowel recognition,SMOTE,0.9753,0.8365,0.9147,0.8717,0.9470
8,Vowel recognition,ADASYN,0.9678,0.7603,0.9560,0.8453,0.9622
9,Ionosphere,Decision tree,0.8617,0.8403,0.7698,0.8003,0.8371


## Load Dataset

In [5]:
if config.should_run_experiment:
    df = load_q2_dataset(config)
    display(df.head())
    display(df["label"].value_counts().rename_axis("label").to_frame("count"))
else:
    df = None
    print("RUN_PROFILE=view: skipping dataset load; saved CSV tables will be loaded below.")


,preg,plas,pres,skin,insu,mass,pedi,age,label
0,6,148,72,35,0,33.6,0.627,50,minority
1,1,85,66,29,0,26.6,0.351,31,majority
2,8,183,64,0,0,23.3,0.672,32,minority
3,1,89,66,23,94,28.1,0.167,21,majority
4,0,137,40,35,168,43.1,2.288,33,minority


,count
label,
majority,500
minority,268


## Run Or Reuse Q2 Benchmark

`run_full` repeats the ADASYN half-split protocol and saves row-level and summary CSVs. `view` skips fitting and uses filenames reconstructed from `DATASET` and the config.

In [ ]:
if config.should_run_experiment:
    run_q2_benchmark(df, config, progress=print_progress_event)
else:
    print("RUN_PROFILE=view: skipping experiment run.")

print("Q2 result table filenames:")
for table, path in q2_result_table_paths(config).items():
    print(f"{table}: {path}")
display(q2_result_table_manifest(config))


Starting Q2 runs: 0/100 complete


## Load Saved Result Tables

Performance summaries and plots below read from the CSV tables saved by the benchmark.

In [ ]:
run_results, q2_summary, manuscript_table = load_q2_result_tables(config)

display(q2_summary)
display(run_results.head())
display(manuscript_table)


## Plot Metric Comparison

In [ ]:
fig, ax = plot_q2_metric_comparison(manuscript_table, metric="g_mean")
